# Grafana and PEFT Playbook

This notebook covers two useful additions to an agentic AI toolkit:

- Grafana for observability, metrics, traces, and alerting
- PEFT for parameter-efficient fine-tuning of models such as LoRA-style adaptation


## Why these matter

Most agentic systems fail in practice because they are hard to observe or too expensive to adapt. Grafana helps you see what the system is doing. PEFT helps you adapt a model without full fine-tuning cost.


## Grafana tips

Grafana is useful when you want to query, visualize, alert on, and explore metrics, logs, and traces. For agentic AI, watch things like:

- request latency
- tool-call count
- retrieval hit rate
- fallback rate
- human escalation rate
- cost per request
- eval pass rate


In [ ]:
observability_snapshot = {
    'service': 'agent-copilot',
    'latency_ms': 840,
    'tool_calls': 3,
    'retrieval_hit_rate': 0.92,
    'fallback_rate': 0.08,
    'human_escalations': 1,
    'cost_per_request_usd': 0.0041,
}
observability_snapshot


## A simple dashboard mindset

A good dashboard is not just a wall of charts. It should help you answer:

- Is the agent fast enough?
- Is it safe enough?
- Is it cheap enough?
- Is it useful enough?


In [ ]:
def dashboard_fields(snapshot: dict) -> list[str]:
    return [
        f"latency={snapshot['latency_ms']} ms",
        f"tool_calls={snapshot['tool_calls']}",
        f"fallback_rate={snapshot['fallback_rate']:.2%}",
        f"escalations={snapshot['human_escalations']}",
    ]

dashboard_fields(observability_snapshot)


## Alerting trick

Alert on symptoms, not just raw errors. For example, alert when latency climbs and retrieval quality drops together.


In [ ]:
def alert_policy(snapshot: dict) -> str:
    if snapshot['latency_ms'] > 1000 and snapshot['retrieval_hit_rate'] < 0.85:
        return 'alert: agent is slow and underperforming'
    if snapshot['fallback_rate'] > 0.2:
        return 'alert: too many weak-context fallbacks'
    return 'ok'

alert_policy(observability_snapshot)


## PEFT tips

PEFT stands for Parameter-Efficient Fine-Tuning. The main idea is to adapt a pretrained model by updating a small number of parameters instead of retraining everything. That keeps compute and storage much lower.


## When to use PEFT

- You have a domain-specific task and a smaller dataset
- You want better quality than prompting alone
- You cannot afford full fine-tuning
- You want to swap adapters instead of maintaining many full models


In [ ]:
from dataclasses import dataclass

@dataclass
class PeftPlan:
    base_model: str
    method: str
    dataset_size: str
    notes: str

def choose_peft_plan(task: str) -> PeftPlan:
    lowered = task.lower()
    if 'classification' in lowered or 'routing' in lowered:
        return PeftPlan('small instruct model', 'LoRA', 'small', 'Good for fast domain adaptation')
    if 'generation' in lowered or 'assistant' in lowered:
        return PeftPlan('mid-size instruct model', 'LoRA or QLoRA', 'medium', 'Useful when style and domain tone matter')
    return PeftPlan('pretrained base model', 'LoRA', 'small-to-medium', 'Start simple and measure before scaling up')

choose_peft_plan('Need a domain assistant for customer support')


## Practical PEFT tricks

- Freeze the base model and train adapters first
- Use a small, clean dataset
- Track whether PEFT actually beats prompting
- Merge adapters only after you validate the gains
- Keep a non-PEFT fallback path for demos


In [ ]:
def peft_decision(use_case: str, data_size: int) -> str:
    if data_size < 500:
        return 'prompting or light PEFT'
    if data_size < 5000:
        return 'LoRA / QLoRA'
    return 'consider PEFT first, then compare to full fine-tuning if quality is still insufficient'

peft_decision('domain assistant', 1800)


## Combined workflow idea

A strong pattern is to use Grafana to watch the system and PEFT to improve the model behind it. That gives you a closed loop:

1. measure performance
2. identify weakness
3. adapt the model
4. re-measure


In [ ]:
closed_loop = {
    'observe': 'Grafana dashboard and alerts',
    'adapt': 'PEFT adapter training',
    'validate': 'eval dataset and human review',
    'ship': 'merge adapters or keep them separate for deployment',
}
closed_loop


## Serving tip

If the adapted model needs better throughput later, a GPU-serving layer such as vLLM can sit behind the same application interface.


## Final checklist

- keep the observability signal simple
- alert on agent quality, not just infra failures
- prefer LoRA/QLoRA before full fine-tuning
- compare against a prompting baseline
- keep a fallback model path for demos and production
